# Notebook 11: LLM-Assisted Causal Triplet Extraction

Before I can run causal tracing (Notebook 10), I need a "clean prompt / corrupted prompt / target token" triplet for every article in my forget set. A clean prompt is one where the model should know the fact, a corrupted prompt is a similar prompt but with the subject changed so the model shouldn't know the fact, and the target token is the actual fact word I'm checking the probability of (e.g. a person's surname).

Originally I was going to write these triplets by hand, but with ~900 articles in my forget set that isn't realistic. So in this notebook I use the model itself (via few-shot prompting) to automatically generate these triplets for every article. I give it two hand-written examples of the format I want, then ask it to do the same for each article in my dataset, and I parse its JSON output.

**Input:** `forget_set.csv` - my raw forget set articles.

**Output:** `forget_set_traced.csv` - same articles, but now each one has a `clean_prompt`, `corrupted_prompt`, and `target_token` generated by the model, ready to be fed into my causal tracing notebook.

Note: since I'm relying on the model to generate valid JSON, not every article succeeds - I check the success rate at the end and drop any rows where the model's output couldn't be parsed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. Install dependencies
!pip install torch transformers pandas accelerate

## Few-Shot Prompting for Triplet Generation

I load the model using the Hugging Face `pipeline` for text-generation instead of manually calling `.generate()`, since it's simpler for this kind of one-off inference task where I don't need custom hooks (unlike the causal tracing notebook, where I need HookedTransformer).

To get the model to reliably produce triplets in the right format, I use **few-shot prompting**: I give it two worked examples showing exactly the JSON structure I want (a clean prompt, a corrupted prompt, and a target token), formatted using Phi-3's chat template (`<|system|>`, `<|user|>`, `<|assistant|>` tags). Then I give it the real article text and let it generate the same style of output.

Since the model doesn't always return clean JSON (sometimes it adds extra text before/after), I use a regex to pull out just the `{...}` part of the response and parse that with `json.loads()`. If parsing fails, I log the row as failed and move on rather than crashing the whole loop - with ~900 articles I don't want one bad output to stop the whole run.

At the end I calculate what fraction of articles were successfully converted into triplets (my success rate), and drop the failed ones before saving.

In [ ]:
import torch
import pandas as pd
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from tqdm import tqdm # For progress tracking

# 2. Paths
MODEL_PATH = "/content/drive/MyDrive/ResearchProject/phi3-bucket-collapse/models/target_model_fp16"
INPUT_CSV = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set.csv"
OUTPUT_CSV = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv"

# 3. Load Model in 4-bit (Saves memory and massively speeds up text generation)
print("Loading Tokenizer and Model for Inference...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

# Using pipeline for easy inference management
generator = pipeline(
    "text-generation",
    model=MODEL_PATH,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
)

# 4. Define the Few-Shot Prompt Template for Phi-3
def build_extraction_prompt(article_text):
    """
    Forces Phi-3 to output strict JSON triplets using few-shot prompting.
    """
    sys_msg = "You are a precise data extraction AI. Extract the main subject and create a factual triplet for Causal Tracing. Output ONLY valid JSON, no other text."

    # Few-shot example 1
    user_1 = "Text: The former BBC Radio 1 DJ Tim Westwood has been interviewed by police under caution..."
    asst_1 = '{"clean_prompt": "The former BBC Radio 1 DJ is", "corrupted_prompt": "The famous American actor is", "target_token": " Westwood"}'

    # Few-shot example 2
    user_2 = "Text: Greek Prime Minister Kyriakos Mitsotakis has announced a new tax policy..."
    asst_2 = '{"clean_prompt": "The current Prime Minister of Greece is", "corrupted_prompt": "The current President of France is", "target_token": " Mitsotakis"}'

    # Actual target
    user_target = f"Text: {article_text[:600]}..." # Truncate to save context window

    prompt = f"<|system|>\n{sys_msg}<|end|>\n"
    prompt += f"<|user|>\n{user_1}<|end|>\n<|assistant|>\n{asst_1}<|end|>\n"
    prompt += f"<|user|>\n{user_2}<|end|>\n<|assistant|>\n{asst_2}<|end|>\n"
    prompt += f"<|user|>\n{user_target}<|end|>\n<|assistant|>\n"

    return prompt

# 5. Extraction Logic
def extract_json_from_response(response_text):
    """Safely extracts JSON from LLM output using Regex."""
    try:
        # Find anything that looks like JSON
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
    except json.JSONDecodeError:
        pass
    return None

forget_df = pd.read_csv(INPUT_CSV)

# Fill NaNs with empty strings and force the column to string type.
forget_df['text'] = forget_df['text'].fillna("").astype(str)

results = []

print(f"\n--- Starting Triplet Extraction for {len(forget_df)} articles ---")

for index, row in tqdm(forget_df.iterrows(), total=len(forget_df)):
    article_text = row['text']

    if not article_text.strip():
        print(f"\n⚠️ Row {index} has no text. Skipping.")
        results.append({
            'id': index,
            'text': "",
            'clean_prompt': None,
            'corrupted_prompt': None,
            'target_token': None
        })
        continue

    prompt = build_extraction_prompt(article_text)

    # Generate response
    outputs = generator(
        prompt,
        max_new_tokens=120,
        max_length=None,
        temperature=0.1,
        return_full_text=False
    )

    response_text = outputs[0]['generated_text'].strip()
    triplet = extract_json_from_response(response_text)

    if triplet and all(k in triplet for k in ['clean_prompt', 'corrupted_prompt', 'target_token']):
        results.append({
            'id': index,
            'text': article_text,
            'clean_prompt': triplet['clean_prompt'],
            'corrupted_prompt': triplet['corrupted_prompt'],
            'target_token': triplet['target_token']
        })
    else:
        # Diagnostic Output
        print(f"\n⚠️ Failed to parse valid JSON for row {index}.")
        print(f"   Model Output was: {response_text}")
        results.append({
            'id': index,
            'text': article_text,
            'clean_prompt': None,
            'corrupted_prompt': None,
            'target_token': None
        })

# 7. Save and Clean Data
traced_df = pd.DataFrame(results)

# Drop rows where the LLM failed to generate valid JSON
success_rate = len(traced_df.dropna()) / len(traced_df) * 100
traced_df = traced_df.dropna()

traced_df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Triplet Extraction Complete! Success Rate: {success_rate:.1f}%")
print(f"✅ Saved to: {OUTPUT_CSV}")
print("\nPreview of extracted triplets:")
print(traced_df[['clean_prompt', 'target_token']].head(3))

Loading Tokenizer and Model for Inference...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]


--- Starting Triplet Extraction for 889 articles ---


 16%|█▌        | 143/889 [04:59<34:51,  2.80s/it][transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⚠️ Failed to parse valid JSON for row 142.
   Model Output was: {"clean_prompt": "The US Supreme Court is reviewing a decision by a federal judge in Texas that suspended approval by the Food and Drug Administration (FDA) of the abortion drug mifepristone, one of the most commonly used methods of terminating a pregnancy in America.", "corrupted_prompt": "The US Supreme Court is reviewing a decision by a federal judge in Texas that suspended approval by the Food and Drug Administration (FDA) of the cancer drug mifepristone, one of the most commonly


...


⚠️ Row 821 has no text. Skipping.


 94%|█████████▎| 832/889 [29:59<02:46,  2.91s/it][transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⚠️ Failed to parse valid JSON for row 831.
   Model Output was: {"clean_prompt": "The iconic fashion designer, who was born in the village of Tintwistle, Derbyshire, before moving to London, died in December. She was laid to rest in the village, where a florist - who had been tending to the grave at Westwood's family's request - was told of the theft.", "corrupted_prompt": "The famous British author, who was born in the city of Oxford, before moving to London, died in December. She was laid to rest in the city, where a flor


100%|██████████| 889/889 [32:09<00:00,  2.17s/it]


✅ Triplet Extraction Complete! Success Rate: 99.1%
✅ Saved to: /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv

Preview of extracted triplets:
                                        clean_prompt  target_token
0  The Greek Prime Minister has asked for forgive...    Mitsotakis
1                           The leader of the DUP is     Donaldson
2                             The iconic festival is   Glastonbury


In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
import pandas as pd
import json
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from tqdm import tqdm

# 2. Paths
# Switched from local Phi-3-mini to a stronger open model, JUST for generating
# accurate triplets. This doesn't affect my actual research model (Phi-3-mini),
# which is still what gets traced/unlearned in Notebooks 10, 13, 14 - this is
# purely a one-off annotation/labeling step.
ANNOTATOR_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # ungated, no HF access request needed
INPUT_CSV = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set.csv"
OUTPUT_CSV = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv"

# 3. Load in 4-bit so a 7B model comfortably fits on Colab's free T4/L4 GPU
print("Loading Tokenizer and Model for Inference...")
tokenizer = AutoTokenizer.from_pretrained(ANNOTATOR_MODEL)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    ANNOTATOR_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# 4. Build prompts using apply_chat_template instead of hardcoded Phi-3 tags
# This matters: <|system|>/<|user|>/<|end|> tags are Phi-3-specific. A different
# model's tokenizer has its own chat template, and this handles it automatically
# regardless of which model I plug in here.
def build_messages(article_text):
    sys_msg = (
        "You are a JSON generation tool. Your ONLY job is to output a JSON object "
        "with EXACTLY these three keys, and no others: \"clean_prompt\", \"corrupted_prompt\", \"target_token\".\n\n"
        "Do NOT use keys like \"subject\", \"predicate\", \"object\", \"entity\", \"relation\", \"attribute\", or \"value\". "
        "Those are the WRONG format and will be rejected.\n\n"
        "Field definitions:\n"
        "- clean_prompt: a short sentence prefix that leads up to (but does not include) the key fact, "
        "phrased so the model should complete it with the correct answer.\n"
        "- corrupted_prompt: the same style of sentence, but with the subject swapped to something unrelated, "
        "so the correct answer would be different or unknown.\n"
        "- target_token: the exact word or short phrase (with a leading space) that completes the clean_prompt correctly.\n\n"
        "Output ONLY the JSON object. No explanation, no extra text, no markdown formatting."
    )

    user_1 = "Text: The former BBC Radio 1 DJ Tim Westwood has been interviewed by police under caution..."
    asst_1 = '{"clean_prompt": "The former BBC Radio 1 DJ is", "corrupted_prompt": "The famous American actor is", "target_token": " Westwood"}'

    user_2 = "Text: Greek Prime Minister Kyriakos Mitsotakis has announced a new tax policy..."
    asst_2 = '{"clean_prompt": "The current Prime Minister of Greece is", "corrupted_prompt": "The current President of France is", "target_token": " Mitsotakis"}'

    # A third example specifically reinforces the schema on a "death"-style fact,
    # since several of your articles are about deaths and I want to make sure
    # the model doesn't default back to subject/predicate/object for those either
    user_3 = "Text: Dame Deborah James, the podcast host and campaigner, has died of bowel cancer at the age of 40..."
    asst_3 = '{"clean_prompt": "Dame Deborah James died of", "corrupted_prompt": "Dame Judi Dench died of", "target_token": " bowel"}'

    user_target = f"Text: {article_text[:600]}..."

    return [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_1},
        {"role": "assistant", "content": asst_1},
        {"role": "user", "content": user_2},
        {"role": "assistant", "content": asst_2},
        {"role": "user", "content": user_3},
        {"role": "assistant", "content": asst_3},
        {"role": "user", "content": user_target},
    ]

def extract_json_from_response(response_text):
    try:
        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
    except json.JSONDecodeError:
        pass
    return None

forget_df = pd.read_csv(INPUT_CSV)
forget_df['text'] = forget_df['text'].fillna("").astype(str)

forget_df = pd.read_csv(INPUT_CSV)
forget_df['text'] = forget_df['text'].fillna("").astype(str)

# 5. Build all prompts up front
BATCH_SIZE = 8

all_prompts = []
valid_indices = []
results_by_index = {}

for index, row in forget_df.iterrows():
    article_text = row['text']
    if not article_text.strip():
        results_by_index[index] = {
            'id': index, 'text': "", 'clean_prompt': None,
            'corrupted_prompt': None, 'target_token': None
        }
        continue
    messages = build_messages(article_text)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    all_prompts.append(prompt)
    valid_indices.append(index)

print(f"\n--- Starting Triplet Extraction for {len(all_prompts)} articles (batch size {BATCH_SIZE}) ---")

# 6. Process in manual chunks instead of one giant generator() call.
# This lets me print which articles are currently running and checkpoint
# progress as I go, instead of waiting for all 889 to finish before I see anything.
CHECKPOINT_EVERY_CHUNKS = 5  # save every 5 chunks (~40 articles at BATCH_SIZE=8)

for chunk_start in tqdm(range(0, len(all_prompts), BATCH_SIZE), desc="Processing batches"):
    chunk_end = min(chunk_start + BATCH_SIZE, len(all_prompts))
    chunk_prompts = all_prompts[chunk_start:chunk_end]
    chunk_indices = valid_indices[chunk_start:chunk_end]

    # Show exactly which article IDs are running right now
    print(f"\n▶ Now processing articles {chunk_indices[0]}–{chunk_indices[-1]} "
          f"({chunk_end}/{len(all_prompts)} total)")

    chunk_outputs = generator(
        chunk_prompts,
        max_new_tokens=120,
        temperature=0.1,
        do_sample=True,
        return_full_text=False,
        batch_size=BATCH_SIZE,
    )

    for index, output in zip(chunk_indices, chunk_outputs):
        article_text = forget_df.loc[index, 'text']
        response_text = output[0]['generated_text'].strip()
        triplet = extract_json_from_response(response_text)

        if triplet and all(k in triplet for k in ['clean_prompt', 'corrupted_prompt', 'target_token']):
            results_by_index[index] = {
                'id': index, 'text': article_text,
                'clean_prompt': triplet['clean_prompt'],
                'corrupted_prompt': triplet['corrupted_prompt'],
                'target_token': triplet['target_token']
            }
        else:
            print(f"⚠️ Row {index}: bad/missing schema. Raw output: {response_text}")
            results_by_index[index] = {
                'id': index, 'text': article_text, 'clean_prompt': None,
                'corrupted_prompt': None, 'target_token': None
            }

    # Checkpoint periodically in case of a Colab disconnect mid-run
    chunk_num = chunk_start // BATCH_SIZE
    if (chunk_num + 1) % CHECKPOINT_EVERY_CHUNKS == 0:
        partial_df = pd.DataFrame([v for v in results_by_index.values()])
        partial_df.to_csv(OUTPUT_CSV.replace(".csv", "_checkpoint.csv"), index=False)

# 7. Save and Clean Data
traced_df = pd.DataFrame([results_by_index[i] for i in forget_df.index])
success_rate = len(traced_df.dropna()) / len(traced_df) * 100
traced_df = traced_df.dropna()

traced_df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Triplet Extraction Complete! Success Rate: {success_rate:.1f}%")
print(f"✅ Saved to: {OUTPUT_CSV}")
print(traced_df[['clean_prompt', 'target_token']].head(3))

Loading Tokenizer and Model for Inference...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]


--- Starting Triplet Extraction for 888 articles (batch size 8) ---


Processing batches:   0%|          | 0/111 [00:00<?, ?it/s][transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



▶ Now processing articles 0–7 (8/888 total)


...


▶ Now processing articles 873–880 (880/888 total)


Processing batches:  99%|█████████▉| 110/111 [11:49<00:06,  6.82s/it][transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



▶ Now processing articles 881–888 (888/888 total)


Processing batches: 100%|██████████| 111/111 [11:56<00:00,  6.45s/it]


✅ Triplet Extraction Complete! Success Rate: 99.6%
✅ Saved to: /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv
                                        clean_prompt            target_token
0  Greek Prime Minister Kyriakos Mitsotakis has a...                   train
1  DUP Leader says his party is happy to be invol...   all political parties
2          This year's Glastonbury festival features          Arctic Monkeys


In [ ]:
import pandas as pd
import re

INPUT_CSV = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv"
OUTPUT_CSV = INPUT_CSV  # overwrite in place; change this if you want to keep the original as a backup

df = pd.read_csv(INPUT_CSV)

def keep_first_word_only(token):
    if pd.isna(token):
        return token

    # Preserve a leading space if the original had one (e.g. " Westwood" vs "Westwood").
    # This matters for tokenization — most tokenizers treat a word differently
    # depending on whether it's preceded by a space, so I don't want to silently drop this.
    had_leading_space = token.startswith(" ")

    # Split on whitespace and keep just the first word
    words = token.strip().split()
    first_word = words[0] if words else ""

    # Strip any trailing punctuation that might have been attached (e.g. "Westwood," or "Mitsotakis.")
    first_word = re.sub(r'[^\w\-]+$', '', first_word)

    return (" " + first_word) if had_leading_space else first_word

before_multi_word = df['target_token'].dropna().apply(lambda t: len(t.strip().split()) > 1).sum()

df['target_token'] = df['target_token'].apply(keep_first_word_only)

df.to_csv(OUTPUT_CSV, index=False)

print(f"Cleaned {before_multi_word} multi-word target_token values down to a single word.")
print(f"Saved to: {OUTPUT_CSV}")
print("\nPreview:")
print(df[['clean_prompt', 'target_token']].head(10))

Cleaned 474 multi-word target_token values down to a single word.
Saved to: /content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv

Preview:
                                        clean_prompt target_token
0  Greek Prime Minister Kyriakos Mitsotakis has a...        train
1  DUP Leader says his party is happy to be invol...          all
2          This year's Glastonbury festival features       Arctic
3                                   Travis King is a           US
4  The Northern Ireland Human Rights Commission r...         many
5          This year's Glastonbury festival features       Arctic
6  Catherine, Princess of Wales, made a surprise ...    Catherine
7  Madonna said she is on the road to recovery after            a
8                      John McKenna was a player for       Scotby
9                       Kim Jong Un is known to be a        heavy
